# 🫧 K-Means Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> K-means groups customers into segments by finding cluster centers that minimize within-cluster distance. The result: actionable customer segments you can market to differently.

### Dataset
**UCI Online Retail — Customer RFM Analysis**
Real e-commerce transaction data from a UK gift retailer (Chen et al. 2012, UCI ML Repository).
4,372 customers. RFM features derived from actual purchase behaviour:
- **Recency** — days since last purchase
- **Frequency** — number of distinct orders
- **Monetary** — total spend (£)

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess, os, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
print("\u2713 Setup complete")

In [ ]:
# Load UCI Online Retail RFM dataset from course repo
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)
DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'

rfm = pd.read_csv(f'{DATA_DIR}/OnlineRetailRFM.csv')

print("UCI Online Retail — Customer RFM Dataset")
print(f"Source: Chen et al. 2012, UCI ML Repository")
print(f"4,372 unique customers, UK gift retailer, Dec 2010 - Dec 2011")
print(f"\nRFM distributions:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(1).to_string())
print(f"\nInterpretation:")
print(f"  Recency:   days since last purchase  (lower = more recent = better)")
print(f"  Frequency: number of distinct orders  (higher = better)")
print(f"  Monetary:  total spend in GBP  (higher = better)")
print(f"\n  Recency median {rfm.Recency.median():.0f}d — half of customers haven't bought in 3+ months")
print(f"  Frequency median {rfm.Frequency.median():.0f} orders — most customers buy only once or twice")
print(f"  Monetary median \xa3{rfm.Monetary.median():,.0f} — strong right skew, a few high-value customers")

## 📐 Part 1 — Why RFM? The Business Logic

RFM is the gold standard framework for customer analytics because it captures three dimensions that predict future behaviour:

| Dimension | What it measures | Why it matters |
|-----------|-----------------|----------------|
| **Recency** | How recently did they buy? | Recent buyers are more likely to buy again |
| **Frequency** | How often do they buy? | Frequent buyers have stronger loyalty |
| **Monetary** | How much do they spend? | High spenders generate disproportionate revenue |

**The Pareto principle in retail:** typically 20% of customers generate 80% of revenue. RFM clustering finds that 20%.

**K-means** assigns each customer to the nearest cluster centroid and updates centroids iteratively. We must standardize first — £Monetary would completely dominate days-Recency otherwise.

In [ ]:
# Log-transform Monetary (heavy right skew) then standardize all features
rfm_model = rfm.copy()
rfm_model['Monetary_log'] = np.log1p(rfm_model['Monetary'])
features = ['Recency', 'Frequency', 'Monetary_log']

X = rfm_model[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Invert Recency so higher score always = better (for intuitive segment naming)
X_scaled[:, 0] *= -1

# Show why standardization matters
print("=== Why we must standardize ===\n")
print(f"{'Feature':<15} {'Raw range':<20} {'Std range':<20}")
print("-"*55)
for i, feat in enumerate(features):
    raw  = X[:, i]
    std  = X_scaled[:, i]
    print(f"{feat:<15} [{raw.min():>6.0f}, {raw.max():>6.0f}]    [{std.min():>5.2f}, {std.max():>5.2f}]")

print("\n  Without scaling: Monetary (0-80,000) dominates Recency (1-365)")
print("  With scaling: all three features contribute equally to distance")

In [ ]:
# Elbow + Silhouette to choose K
wcss = []
sil  = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels, sample_size=1000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), wcss, 'o-', color='#1e5fa8', lw=2, markersize=7)
axes[0].set_xlabel('Number of Segments (K)')
axes[0].set_ylabel('WCSS')
axes[0].set_title('Elbow Method — look for the bend')

best_k = list(K_range)[sil.index(max(sil))]
axes[1].plot(list(K_range), sil, 's-', color='#e85d20', lw=2, markersize=7)
axes[1].axvline(best_k, color='#1a7a45', lw=1.5, ls='--',
                label=f'Best K={best_k}  (silhouette={max(sil):.3f})')
axes[1].set_xlabel('Number of Segments (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher = more distinct segments')
axes[1].legend()

plt.suptitle('Choosing K — UCI Online Retail Customers', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print(f"\n\u2714 Silhouette suggests K={best_k} distinct customer segments")

In [ ]:
# Fit final model
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['Cluster'] = km_final.fit_predict(X_scaled)

# Profile each cluster on ORIGINAL (untransformed) scale
profile = rfm.groupby('Cluster').agg(
    N=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean'),
    Med_Monetary=('Monetary','median'),
).round(1)
profile['Pct'] = (profile['N'] / len(rfm) * 100).round(1)

# Name segments based on profile
def name_seg(row):
    if row['Avg_Recency'] < 40 and row['Avg_Frequency'] >= 6:
        return 'Champions'
    elif row['Avg_Recency'] < 70 and row['Avg_Frequency'] >= 4:
        return 'Loyal Customers'
    elif row['Avg_Recency'] > 200:
        return 'Lost / Inactive'
    elif row['Avg_Frequency'] <= 1.5:
        return 'New / One-Time'
    else:
        return 'At-Risk'

profile['Segment'] = profile.apply(name_seg, axis=1)
rfm['Segment'] = rfm['Cluster'].map(profile['Segment'].to_dict())

print("=== Customer Segment Profiles (UCI Online Retail) ===\n")
display_cols = ['Segment','N','Pct','Avg_Recency','Avg_Frequency','Avg_Monetary','Med_Monetary']
print(profile[display_cols].sort_values('Avg_Monetary', ascending=False).to_string())

print("\n\u2714 Revenue concentration:")
for _, row in profile.sort_values('Avg_Monetary', ascending=False).iterrows():
    rev_share = row['N'] * row['Avg_Monetary'] / (rfm['Monetary'].sum()) * 100
    print(f"  {row['Segment']:<20} {row['Pct']:>5.1f}% of customers  "
          f"\xa3{row['Avg_Monetary']:>8,.0f} avg spend  "
          f"{rev_share:>5.1f}% of total revenue")

In [ ]:
# Visualise segments in PCA space + profile bar chart
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

seg_order = profile.sort_values('Avg_Monetary', ascending=False)['Segment'].tolist()
colors = ['#1a7a45','#1e5fa8','#e85d20','#6b46c1','#0e7490']
seg_color = {s: colors[i] for i, s in enumerate(seg_order)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA scatter
for seg in seg_order:
    mask = rfm['Segment'] == seg
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=8, alpha=0.5, color=seg_color[seg], label=seg)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% variance)')
axes[0].set_title('Customer Segments in PCA Space\n(UCI Online Retail)')
axes[0].legend(fontsize=9, markerscale=3)

# Normalised profile
profile_norm = profile[['Avg_Recency','Avg_Frequency','Avg_Monetary']].copy()
# For Recency: invert so bar = higher is better
profile_norm['Avg_Recency'] = profile_norm['Avg_Recency'].max() - profile_norm['Avg_Recency']
for col in profile_norm.columns:
    mn, mx = profile_norm[col].min(), profile_norm[col].max()
    profile_norm[col] = (profile_norm[col]-mn)/(mx-mn+1e-9)
profile_norm.index = profile['Segment']
profile_norm.columns = ['Recency (inverted)','Frequency','Monetary']
profile_norm.loc[seg_order].plot(kind='bar', ax=axes[1],
    color=['#1e5fa8','#e85d20','#1a7a45'], edgecolor='white', width=0.7)
axes[1].set_title('Normalised Segment Profiles\n(higher = better on all axes)')
axes[1].set_xlabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("\n\u2714 Business actions by segment:")
actions = {
    'Champions':       'Reward and ask for referrals — they are your best advocates',
    'Loyal Customers': 'Upsell and cross-sell — they are engaged and trust you',
    'At-Risk':         'Win-back campaign — personalised discount before they leave',
    'New / One-Time':  'Onboarding nurture — encourage a second purchase within 30 days',
    'Lost / Inactive': 'Low-cost reactivation email — or accept churn and reduce spend',
}
for seg, action in actions.items():
    if seg in rfm['Segment'].values:
        n_seg = (rfm['Segment']==seg).sum()
        print(f"  {seg:<22} ({n_seg:,} customers) — {action}")

In [ ]:
# @title 📝 Quiz — K-Means Clustering
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does K-means minimise, and why must features be standardised first?
# @markdown **Q2:** What does the silhouette score measure? What does a value near 0 mean?
# @markdown **Q3:** What is the elbow method and why doesn't it always give a clear answer?
# @markdown **Q4:** Why is K-means sensitive to initialisation, and how does n_init help?
# @markdown **Q5:** For the Champions segment: describe ONE concrete marketing action and why it fits.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_ = "K-Means Clustering"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*